# Aplicación de LSTM a series de tiempo

## Objetivo

En este notebook veremos una aplicación complementaria de modelos secuenciales, ahora sobre una **serie de tiempo**.

En particular, aprenderemos a:

- descargar una serie temporal simple,
- visualizarla,
- construir ventanas de tiempo,
- entrenar una red `LSTM`,
- evaluar sus predicciones,
- y discutir cómo cambia el problema al usar más de una variable.

La idea principal es mostrar que los modelos secuenciales no se usan solo en texto: también son útiles para datos ordenados en el tiempo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

In [ ]:
!pip -q install yfinance

In [ ]:
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

## Hoja de ruta

Trabajaremos en los siguientes pasos:

1. Descargar una serie temporal.
2. Visualizarla.
3. Preparar los datos.
4. Construir ventanas temporales.
5. Entrenar una LSTM univariada.
6. Evaluar sus predicciones.
7. Ver un bonus con entrada multivariada.

## 1. Descargar una serie temporal

Usaremos datos diarios de una acción bursátil solo como ejemplo de serie temporal.

Lo importante aquí no es el contexto financiero, sino la idea de trabajar con una secuencia numérica ordenada en el tiempo.

In [ ]:
ticker = "HMC"   # Honda Motor Co.
start_date = "2018-01-01"
end_date = "2025-12-31"

df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columnas:", list(df.columns))

Para comenzar de forma simple, usaremos solo la columna `Close`, es decir, el precio de cierre diario.

In [ ]:
series = df[["Close"]].dropna().copy()
series.head()

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(series.index, series["Close"])
plt.title("Serie temporal: precio de cierre diario")
plt.xlabel("Fecha")
plt.ylabel("Close")
plt.show()

## 2. Escalamiento

Como en otros problemas de deep learning, es conveniente escalar los datos antes de entrenar.

Usaremos `MinMaxScaler` para llevar los valores al rango `[0, 1]`.

In [ ]:
scaler = MinMaxScaler()

series_scaled = scaler.fit_transform(series)
series_scaled[:5]

## 3. Construcción de ventanas temporales

Para entrenar una LSTM, no le pasamos toda la serie completa de una vez.

En cambio, construimos ejemplos del tipo:

- entrada: últimos `window_size` valores,
- salida: el valor siguiente.

Por ejemplo, si `window_size = 20`, la red observará 20 días consecutivos para intentar predecir el día siguiente.

In [ ]:
window_size = 20

In [ ]:
def create_sequences(data, window_size=20):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i])
        y.append(data[i])
    return np.array(X), np.array(y)

In [ ]:
X_all, y_all = create_sequences(series_scaled, window_size=window_size)

print("Shape de X_all:", X_all.shape)
print("Shape de y_all:", y_all.shape)

### Observación

La entrada `X_all` tiene tres dimensiones:

- número de ejemplos,
- largo de la ventana,
- número de variables.

Eso es exactamente el formato que esperan muchas capas recurrentes en Keras:

`(batch, timesteps, features)`

In [ ]:
print("Primer ejemplo X:")
print(X_all[0].flatten())

print("\nPrimer valor objetivo y:")
print(y_all[0])

## 4. Separación en entrenamiento y test

En series de tiempo no conviene mezclar aleatoriamente los datos como en muchos problemas supervisados tradicionales.

Aquí respetaremos el orden temporal:
- primera parte para entrenamiento,
- última parte para test.

In [ ]:
train_size = int(len(X_all) * 0.8)

X_train = X_all[:train_size]
y_train = y_all[:train_size]

X_test = X_all[train_size:]
y_test = y_all[train_size:]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

## 5. Modelo LSTM univariado

Construiremos ahora una red simple con:

- una capa `LSTM`,
- y una capa `Dense` final para la predicción numérica.

Como la salida es un valor continuo, este es un problema de **regresión**.

In [ ]:
lstm_model = keras.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.LSTM(32),
    layers.Dense(1)
])

lstm_model.summary()

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

Usaremos `metrics="mae"` porque es una métrica muy interpretable en problemas de regresión:

- mide el error absoluto promedio entre predicción y valor real.

In [ ]:
history_lstm_ts = lstm_model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    shuffle=False
)

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history_lstm_ts.history["loss"], label="train loss")
plt.plot(history_lstm_ts.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("LSTM univariada: entrenamiento")
plt.legend()
plt.show()

## 6. Predicción sobre el conjunto de test

In [ ]:
y_pred_scaled = lstm_model.predict(X_test, verbose=0)

In [ ]:
y_test_inv = scaler.inverse_transform(y_test)
y_pred_inv = scaler.inverse_transform(y_pred_scaled)

In [ ]:
mae_value = mean_absolute_error(y_test_inv, y_pred_inv)
print("MAE en test:", round(mae_value, 4))

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(y_test_inv, label="Valor real")
plt.plot(y_pred_inv, label="Predicción", linestyle="--")
plt.title("Predicción con LSTM sobre test")
plt.xlabel("Tiempo")
plt.ylabel("Close")
plt.legend()
plt.show()

### Interpretación

El modelo no “adivina” perfectamente la serie, pero sí puede capturar parte de su evolución local.

Esto es esperable: en series reales, la predicción suele ser más difícil que en ejemplos didácticos muy limpios.

Lo importante aquí es entender el pipeline:

- serie temporal,
- ventanas,
- tensor 3D,
- LSTM,
- predicción.

## 7. Bonus opcional: entrada multivariada

Hasta ahora usamos una sola variable: `Close`.

Pero en muchos problemas de series de tiempo podemos usar varias variables al mismo tiempo.

Por ejemplo:

- `Open`
- `High`
- `Low`
- `Close`
- `Volume`

Eso transforma la entrada en una secuencia donde cada paso temporal tiene más de una característica.

In [ ]:
features = df[["Open", "High", "Low", "Close", "Volume"]].dropna().copy()
features.head()

In [ ]:
feature_scaler = MinMaxScaler()
features_scaled = feature_scaler.fit_transform(features)

target_scaler = MinMaxScaler()
target_close_scaled = target_scaler.fit_transform(features[["Close"]])

In [ ]:
def create_multivariate_sequences(features_data, target_data, window_size=20):
    X, y = [], []
    for i in range(window_size, len(features_data)):
        X.append(features_data[i-window_size:i])
        y.append(target_data[i])
    return np.array(X), np.array(y)

In [ ]:
X_multi, y_multi = create_multivariate_sequences(
    features_scaled,
    target_close_scaled,
    window_size=window_size
)

print("X_multi shape:", X_multi.shape)
print("y_multi shape:", y_multi.shape)

In [ ]:
train_size_multi = int(len(X_multi) * 0.8)

X_train_multi = X_multi[:train_size_multi]
y_train_multi = y_multi[:train_size_multi]

X_test_multi = X_multi[train_size_multi:]
y_test_multi = y_multi[train_size_multi:]

Ahora cada paso temporal ya no tiene una sola variable, sino varias.

La forma de la entrada será:

- `window_size` pasos de tiempo,
- y `5` características por paso.

In [ ]:
multi_lstm_model = keras.Sequential([
    layers.Input(shape=(window_size, X_train_multi.shape[2])),
    layers.LSTM(32),
    layers.Dense(1)
])

multi_lstm_model.summary()

In [ ]:
multi_lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [ ]:
history_multi_lstm = multi_lstm_model.fit(
    X_train_multi,
    y_train_multi,
    epochs=15,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    shuffle=False
)

In [ ]:
y_pred_multi_scaled = multi_lstm_model.predict(X_test_multi, verbose=0)

y_test_multi_inv = target_scaler.inverse_transform(y_test_multi)
y_pred_multi_inv = target_scaler.inverse_transform(y_pred_multi_scaled)

mae_multi = mean_absolute_error(y_test_multi_inv, y_pred_multi_inv)
print("MAE en test (multivariado):", round(mae_multi, 4))

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(y_test_multi_inv, label="Valor real")
plt.plot(y_pred_multi_inv, label="Predicción multivariada", linestyle="--")
plt.title("LSTM multivariada sobre test")
plt.xlabel("Tiempo")
plt.ylabel("Close")
plt.legend()
plt.show()

### Comentario sobre el bonus

En algunos casos, usar más variables puede ayudar al modelo, porque dispone de más contexto por cada paso temporal.

Sin embargo, también aumenta la complejidad del problema y no garantiza automáticamente mejores resultados.

In [ ]:
print("MAE univariado   :", round(mae_value, 4))
print("MAE multivariado :", round(mae_multi, 4))

## Conclusión

En este notebook vimos una aplicación de `LSTM` a series de tiempo.

Primero trabajamos con una entrada **univariada** y luego vimos, como extensión, una versión **multivariada**.

La idea central es que una serie temporal también puede verse como una secuencia, por lo que los modelos recurrentes son una herramienta natural para este tipo de datos.